# Expert Feedback Retraining Loop

This notebook folds clinician corrections back into the training corpus and launches periodic LoRA fine-tuning runs so the chatbot keeps improving.

In [43]:
import json
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional

BASE_DIR = Path.cwd()
PROCESSED_DIR = BASE_DIR / "processed_data"
FEEDBACK_DIR = BASE_DIR / "expert_feedback"
MODEL_OUTPUT_DIR = BASE_DIR / "model_output"
FEEDBACK_RUNS_DIR = MODEL_OUTPUT_DIR / "feedback_runs"

FEEDBACK_RUNS_DIR.mkdir(exist_ok=True)
FEEDBACK_DIR.mkdir(exist_ok=True)

TRAIN_PATH = PROCESSED_DIR / "processed_train.json"
DEV_PATH = PROCESSED_DIR / "processed_dev.json"
ANNOTATIONS_PATH = FEEDBACK_DIR / "annotations.jsonl"
MERGED_TRAIN_PATH = PROCESSED_DIR / "processed_train_with_feedback.jsonl"


In [44]:
def load_json_list(path: Path) -> List[Dict]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_annotations(path: Path) -> List[Dict]:
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def prepare_feedback_examples(annotations: List[Dict]) -> List[Dict]:
    curated: List[Dict] = []
    for ann in annotations:
        label = ann.get("label")
        patient_query = ann.get("patient_query")
        if not patient_query:
            continue

        if label == "correct":
            answer = ann.get("model_response")
        elif label in {"corrected", "partially_correct"}:
            answer = ann.get("corrected_response") or ann.get("model_response")
        else:
            answer = ann.get("corrected_response")

        if not answer:
            continue

        curated.append(
            {
                "text": f"Patient: {patient_query}\nDoctor: {answer}",
                "patient_query": patient_query,
                "doctor_response": answer,
                "meta": {
                    "source": "expert_feedback",
                    "label": label,
                    "reviewer": ann.get("reviewer"),
                    "timestamp": ann.get("timestamp_utc"),
                },
            }
        )
    return curated


In [45]:
def merge_training_data(base_records: List[Dict], feedback_records: List[Dict]) -> List[Dict]:
    merged: Dict[str, Dict] = {}
    for record in base_records:
        key = record.get("patient_query") or record.get("description")
        if not key:
            key = f"base-{len(merged)}"
        merged[key] = record

    for record in feedback_records:
        key = record.get("patient_query")
        if not key:
            key = f"feedback-{len(merged)}"
        merged[key] = record

    merged_list = list(merged.values())
    merged_list.sort(key=lambda item: item.get("meta", {}).get("timestamp", ""), reverse=False)
    return merged_list


In [46]:
def write_jsonl(path: Path, records: List[Dict]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


base_train = load_json_list(TRAIN_PATH)
annotations = load_annotations(ANNOTATIONS_PATH)
feedback_examples = prepare_feedback_examples(annotations)
merged_train = merge_training_data(base_train, feedback_examples)

write_jsonl(MERGED_TRAIN_PATH, merged_train)

print(f"Base training samples: {len(base_train)}")
print(f"Expert feedback samples added: {len(feedback_examples)}")
print(f"Merged training corpus saved → {MERGED_TRAIN_PATH.relative_to(BASE_DIR)}")


Base training samples: 482
Expert feedback samples added: 1
Merged training corpus saved → processed_data/processed_train_with_feedback.jsonl


In [47]:
DEFAULT_TRAINING_CONFIG = {
    "model_name": "mistralai/Mistral-7B-Instruct-v0.2",
    "lora_rank": 64,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "num_epochs": 3,
    "learning_rate": 2e-4,
    "batch_size": 1,
    "gradient_accumulation_steps": 16,
    "max_sequence_length": 1024,
}

CONFIG_PATH = MODEL_OUTPUT_DIR / "training_config.json"
if CONFIG_PATH.exists():
    with CONFIG_PATH.open("r", encoding="utf-8") as f:
        training_config = json.load(f)
        DEFAULT_TRAINING_CONFIG.update(training_config)

training_config = DEFAULT_TRAINING_CONFIG
print("Training configuration:")
for key, value in training_config.items():
    print(f" - {key}: {value}")


Training configuration:
 - model_name: mistralai/Mistral-7B-Instruct-v0.3
 - lora_rank: 16
 - lora_alpha: 32
 - lora_dropout: 0.05
 - num_epochs: 5
 - learning_rate: 0.0002
 - batch_size: 1
 - gradient_accumulation_steps: 8
 - max_sequence_length: 256
 - training_date: Thu Aug 28 07:04:07 PM IST 2025


In [48]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
import torch


def build_dataset(records: List[Dict]) -> Dataset:
    if not records:
        raise ValueError("Merged training set is empty. Collect expert annotations before retraining.")
    return Dataset.from_list(records)


def prepare_tokenizer(model_name: str, max_length: int) -> AutoTokenizer:
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    tokenizer.truncation_side = "right"

    def tokenize_batch(batch: Dict[str, List[str]]) -> Dict[str, List[int]]:
        inputs = tokenizer(
            batch["text"],
            padding="longest",
            truncation=True,
            max_length=max_length,
        )
        inputs["labels"] = inputs["input_ids"].copy()
        return inputs

    tokenizer.tokenize_batch = tokenize_batch  # type: ignore[attr-defined]
    return tokenizer


def prepare_lora_model(model_name: str, training_cfg: Dict[str, float]) -> AutoModelForCausalLM:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        quantization_config=quant_config,
        trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)

    lora_cfg = LoraConfig(
        r=training_cfg["lora_rank"],
        lora_alpha=training_cfg["lora_alpha"],
        lora_dropout=training_cfg["lora_dropout"],
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()
    return model


def train_with_feedback(run_name: Optional[str] = None, records: Optional[List[Dict]] = None) -> Path:
    run_name = run_name or datetime.utcnow().strftime("feedback-%Y%m%d-%H%M%S")
    output_dir = FEEDBACK_RUNS_DIR / run_name
    output_dir.mkdir(parents=True, exist_ok=True)

    if records is None:
        if "merged_train" in globals():
            records = merged_train
        elif MERGED_TRAIN_PATH.exists():
            records = [json.loads(line) for line in MERGED_TRAIN_PATH.open("r", encoding="utf-8") if line.strip()]
        else:
            raise FileNotFoundError(
                "Merged training data not found. Run the merge step to create processed_train_with_feedback.jsonl."
            )

    dataset = build_dataset(records)
    tokenizer = prepare_tokenizer(training_config["model_name"], training_config["max_sequence_length"])
    tokenized_dataset = dataset.shuffle(seed=42).map(
        tokenizer.tokenize_batch,
        batched=True,
        remove_columns=dataset.column_names,
    )

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    model = prepare_lora_model(training_config["model_name"], training_config)

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=training_config["num_epochs"],
        per_device_train_batch_size=training_config["batch_size"],
        gradient_accumulation_steps=training_config["gradient_accumulation_steps"],
        learning_rate=training_config["learning_rate"],
        logging_steps=10,
        save_strategy="epoch",
        evaluation_strategy="no",
        report_to=[],
        fp16=torch.cuda.is_available(),
        bf16=torch.cuda.is_available(),
        dataloader_pin_memory=False,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=data_collator,
    )

    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return output_dir


In [49]:
RUN_TRAINING = False  # Set to True to launch a new LoRA fine-tuning run

if RUN_TRAINING:
    output_path = train_with_feedback()
    print(f"Feedback-tuned adapters saved to {output_path}")
else:
    print("Training skipped. Set RUN_TRAINING = True after reviewing GPU availability.")


Training skipped. Set RUN_TRAINING = True after reviewing GPU availability.


### Operational checklist

1. Run the uncertainty sampling notebook to refresh `expert_feedback/pending_samples.jsonl`.
2. Collect expert annotations through the Flask UI until you have enough new samples.
3. Execute the merge cell above to rebuild `processed_train_with_feedback.jsonl`.
4. Flip `RUN_TRAINING` to `True` and execute the training cell when you have GPU capacity.
5. After training completes, deploy the new adapter directory under `model_output/feedback_runs/` and archive the run metadata.